In [1]:
!apt-get update -y && apt-get install -y ffmpeg

# (2) Make sure PyAV is installed & current (your log shows it's already 15.1.0)
!pip uninstall -y av && pip install --no-cache-dir av


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,085 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [60.9 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,288 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [5,803 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,373 kB]


In [2]:
!git clone https://github.com/Atze00/MoViNet-pytorch.git

Cloning into 'MoViNet-pytorch'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 302 (delta 1), reused 9 (delta 1), pack-reused 293 (from 1)
Receiving objects: 100% (302/302), 606.13 MiB | 42.05 MiB/s, done.
Resolving deltas: 100% (150/150), done.
Updating files: 100% (33/33), done.


In [3]:
%cd MoViNet-pytorch
!pip install -r tests/test_requirements.txt
!pip install torch torchvision torchaudio opencv-python

/kaggle/working/MoViNet-pytorch
  Cloning https://github.com/Atze00/MoViNet-pytorch.git to /tmp/pip-req-build-t4vp63ph
  Running command git clone --filter=blob:none --quiet https://github.com/Atze00/MoViNet-pytorch.git /tmp/pip-req-build-t4vp63ph
  Resolved https://github.com/Atze00/MoViNet-pytorch.git to commit c2d1edf48fc6c5259707f9d833f22171b4f63493
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Prepare my dataset for movinet, I will split into n clips by 30 frames

In [4]:
# --- shared splitter you can reuse anywhere ---
def video_to_clips(frames, frames_per_clip=30, hop=None, max_clips=None):
    """
    frames: [T, C, H, W] uint8
    returns: [K_i, F, C, H, W]
    """
    F = int(frames_per_clip)
    hop = int(hop) if hop is not None else F
    T, C, H, W = frames.shape

    # pad tail if video shorter than one clip
    if T < F:
        pad = F - T
        tail = frames[-1:].expand(pad, C, H, W)
        frames = torch.cat([frames, tail], dim=0)
        T = frames.shape[0]

    starts = list(range(0, max(T - F, 0) + 1, hop))
    if not starts:
        starts = [0]

    if max_clips is not None and len(starts) > max_clips:
        idxs = torch.linspace(0, len(starts)-1, steps=max_clips).round().to(int).tolist()
        starts = [starts[i] for i in idxs]

    clips = [frames[s:s+F] for s in starts]  # each: [F, C, H, W]
    return torch.stack(clips, dim=0)         # [K_i, F, C, H, W]


In [5]:
import os, glob, random, torch
from torch.utils.data import Dataset
import torchvision.io as io
import torchvision
CLASS_TO_IDX = {"NonFight": 0, "Fight": 1}
VIDEO_EXTS = (".avi")

def list_videos(root_split_dir):  # e.g. root_split_dir="RWF2000/train"
    items = []
    for cls in CLASS_TO_IDX:
        for p in glob.glob(os.path.join(root_split_dir, cls, "**"), recursive=True):
            if os.path.isfile(p) and p.lower().endswith(VIDEO_EXTS):
                items.append((p, CLASS_TO_IDX[cls]))
    items.sort()
    return items

class RWFClipsDynamicDataset(Dataset):
    def __init__(self, root_dir, split="train", frames_per_clip=30, clip_hop=None,
                 transform=None, max_clips=None):
        self.items = list_videos(os.path.join(root_dir, split))  # your helper
        self.frames_per_clip = frames_per_clip
        self.clip_hop = clip_hop
        self.transform = transform
        self.max_clips = max_clips

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        frames, _, _ = io.read_video(path, pts_unit="sec")
        frames = frames.permute(0, 3, 1, 2).to(torch.uint8)

        clips = video_to_clips(frames, self.frames_per_clip, self.clip_hop, self.max_clips)
        if self.transform:
            # transform should accept [F, C, H, W] and handle uint8→float/normalize
            clips = torch.stack([self.transform(c) for c in clips], dim=0)

        return clips, label, path


In [6]:
import torch.nn.functional as F
import torchvision.transforms as T
import torch.optim as optim

# Train transform
train_transform = T.Compose([
    T.ConvertImageDtype(torch.float32),
    T.RandomResizedCrop(172, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.Normalize(mean=(0.45, 0.45, 0.45), std=(0.225, 0.225, 0.225)),
])

# Val/Test
val_transform = T.Compose([
    T.ConvertImageDtype(torch.float32),
    T.Resize(172),
    T.CenterCrop(172),
    T.Normalize(mean=(0.45, 0.45, 0.45), std=(0.225, 0.225, 0.225)),
])

In [7]:
train_dataset = RWFClipsDynamicDataset("/kaggle/input/rwf2000/RWF-2000", transform=train_transform)
val_dataset = RWFClipsDynamicDataset("/kaggle/input/rwf2000/RWF-2000", split="val", transform=val_transform)

In [8]:
from torch.utils.data import DataLoader

def collate_dynamic(batch):
    clips_list, labels, paths = zip(*batch)
    return list(clips_list), torch.tensor(labels, dtype=torch.long), list(paths)

batch_size=8

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_dynamic)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_dynamic)

In [9]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
def _to_model_input(x, expects="BFCHW"):
    """
    x is a single clip: [1, F, C, H, W]
    - if your model expects [B, F, C, H, W] -> set expects="BFCHW" (default) and return as is
    - if your model expects [B, C, F, H, W] -> set expects="BCTHW"
    """
    if expects == "BFCHW":
        return x
    elif expects == "BCTHW":
        return x.permute(0, 2, 1, 3, 4).contiguous()
    else:
        raise ValueError("expects must be 'BFCHW' or 'BCTHW'")



def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    expects="BCTHW",
    use_amp=True,
    accum_steps=1,
):
    model.train()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    total_loss, correct, total = 0.0, 0, 0
    leftover_steps = 0

    pbar = tqdm(loader, desc="Train", leave=False)

    optimizer.zero_grad(set_to_none=True)

    for step, (clips_list, labels, _) in enumerate(pbar, 1):
        labels = labels.to(device, non_blocking=True)
        batch_loss = 0.0
        batch_preds = []

        assert isinstance(clips_list, (list, tuple)), f"Expected list of clips, got {type(clips_list)}"
        B = len(clips_list)
        assert B == labels.shape[0], f"Batch mismatch: {B=}, labels={labels.shape}"

        for i, clips in enumerate(clips_list):
            assert clips.dim() == 5, f"Expected [K,F,C,H,W], got {clips.shape}"
            K = clips.shape[0]
            fight_probs = []


            if hasattr(model, "clean_activation_buffers"):
                model.clean_activation_buffers()

            for t in range(K):
                x = clips[t:t+1]  # [1, F, C, H, W]
                if expects == "BCTHW":
                    x = x.permute(0, 2, 1, 3, 4).contiguous()
                x = x.to(device, non_blocking=True).float() / 255.0

                with torch.amp.autocast("cuda", enabled=use_amp):
                    logits = model(x)
                    probs  = logits.softmax(dim=1)
                fight_probs.append(probs[:, 1])

            fight_probs = torch.stack(fight_probs, dim=0)
            max_fight  = fight_probs.max()

            agg_probs = torch.stack([1.0 - max_fight, max_fight]).unsqueeze(0)
            agg_log_probs = torch.log(agg_probs.clamp_min(1e-8))
            loss_i = F.nll_loss(agg_log_probs, labels[i:i+1])

            batch_loss = batch_loss + loss_i
            batch_preds.append(int(torch.argmax(agg_probs, dim=1).item()))

        batch_loss = batch_loss / B
        # Grad accumulation
        batch_loss_scaled = batch_loss / accum_steps
        scaler.scale(batch_loss_scaled).backward()
        leftover_steps += 1

        if leftover_steps % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            leftover_steps = 0

        total_loss += batch_loss.item()
        preds = torch.tensor(batch_preds, device=device)
        correct += (preds == labels).sum().item()
        total += B

        pbar.set_postfix({"loss": f"{batch_loss.item():.4f}", "acc": f"{correct / max(1, total):.3f}"})

    if leftover_steps % accum_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    pbar.close()
    return total_loss / max(1, len(loader)), correct / max(1, total)


@torch.no_grad()
def eval_one_epoch(model, loader, device, expects="BCTHW", use_amp=True):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc="🔵 Eval", leave=False)
    for clips_list, labels, _ in pbar:
        labels = labels.to(device, non_blocking=True)
        batch_loss, batch_preds = 0.0, []

        # (optional) clear streaming buffers per batch/video if your model has them
        if hasattr(model, "clean_activation_buffers"):
            model.clean_activation_buffers()

        for i, clips in enumerate(clips_list):
            fight_probs = []                      # store only P(fight) per clip
            K = clips.shape[0]

            for t in range(K):
                x = clips[t:t+1]                  # [1, F, C, H, W] on CPU
                if expects == "BCTHW":
                    x = x.permute(0, 2, 1, 3, 4).contiguous()  # -> [1, C, F, H, W]
                x = x.to(device, non_blocking=True).float() / 255.0

                with torch.amp.autocast("cuda", enabled=use_amp):
                    logits = model(x)             # [1, 2]
                    probs  = logits.softmax(dim=1)

                fight_probs.append(probs[:, 1])   # keep only class-1 ("Fight")
                del x, logits, probs

            fight_probs = torch.stack(fight_probs, dim=0)   # [K, 1]
            max_fight  = fight_probs.max()                  # scalar

            agg_probs = torch.stack([1.0 - max_fight, max_fight]).unsqueeze(0)  # [1, 2]
            loss_i = F.nll_loss((agg_probs + 1e-8).log(), labels[i:i+1])

            batch_loss += float(loss_i.item())
            batch_preds.append(int(torch.argmax(agg_probs, dim=1).item()))

            del fight_probs, max_fight, agg_probs, loss_i

        batch_loss /= len(clips_list)
        total_loss += batch_loss
        preds = torch.tensor(batch_preds, device=device)
        correct += (preds == labels).sum().item()
        total += len(labels)

        pbar.set_postfix(loss=f"{batch_loss:.4f}", acc=f"{correct/total:.3f}")
        torch.cuda.empty_cache()

    return total_loss / max(1, len(loader)), correct / max(1, total)

In [10]:
def save_ckpt(model, optimizer, tag, out_dir="checkpoints", extra=None):
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{tag}.pt")
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "tag": tag,
        "saved_at": int(time.time()),
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)
    print(f"saved: {path}")
    return path

In [11]:
import time
from movinets import MoViNet
from movinets.config import _C

N_EPOCHS = 10


LR = 5e-5
DEVICE = "cuda"
os.makedirs("checkpoints", exist_ok=True)

model = MoViNet(_C.MODEL.MoViNetA0, causal = True, pretrained = True )
model.to(DEVICE)

trloss_val, tsloss_val = [], []

model.classifier[3] = torch.nn.Conv3d(2048, 2, (1,1,1)).to(DEVICE)
optimz = optim.Adam(model.parameters(), lr=0.00005)
for epoch in range(1, N_EPOCHS + 1):
    print('Epoch:', epoch)
    train_loss, train_acc = train_one_epoch(model, train_loader, optimz, DEVICE)
    val_loss, val_acc = eval_one_epoch(model, val_loader, DEVICE)
    print(f"📊 Epoch {epoch} — "
          f"train loss {train_loss:.4f}, acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f}, acc {val_acc:.3f}")
    save_ckpt(
        model, optimz, tag=f"epoch{epoch:03d}",
        extra={"train_loss": float(train_loss), "val_loss": float(val_loss),
               "train_acc": float(train_acc),  "val_acc": float(val_acc)}
    )


Downloading: "https://github.com/Atze00/MoViNet-pytorch/blob/main/weights/modelA0_stream_statedict_v3?raw=true" to /root/.cache/torch/hub/checkpoints/modelA0_stream_statedict_v3
100%|██████████| 14.5M/14.5M [00:00<00:00, 145MB/s]


Epoch: 1


📊 Epoch 1 — train loss 0.6342, acc 0.664 | val loss 0.7151, acc 0.500
saved: checkpoints/epoch001.pt
Epoch: 2


📊 Epoch 2 — train loss 0.5043, acc 0.757 | val loss 0.7262, acc 0.525
saved: checkpoints/epoch002.pt
Epoch: 3


📊 Epoch 3 — train loss 0.4588, acc 0.790 | val loss 0.7530, acc 0.525
saved: checkpoints/epoch003.pt
Epoch: 4


...

📊 Epoch 6 — train loss 0.3187, acc 0.865 | val loss 1.1116, acc 0.545
saved: checkpoints/epoch006.pt
Epoch: 7


📊 Epoch 7 — train loss 0.2783, acc 0.884 | val loss 1.0380, acc 0.550
saved: checkpoints/epoch007.pt
Epoch: 8


📊 Epoch 8 — train loss 0.2337, acc 0.904 | val loss 1.4242, acc 0.560
saved: checkpoints/epoch008.pt
Epoch: 9


📊 Epoch 9 — train loss 0.2143, acc 0.915 | val loss 0.8538, acc 0.650
saved: checkpoints/epoch009.pt
Epoch: 10


📊 Epoch 10 — train loss 0.1969, acc 0.920 | val loss 1.6890, acc 0.605
saved: checkpoints/epoch010.pt
